In [2]:
import pandas as pd
import matplotlib as plt

In [3]:
df = pd.read_csv('zeroshot_qwen3_30b.csv')
df.head(3)

,Problem,Level,Hardware,Device,Compiled,Correctness,Ref_PyTorch_Runtime_ms,Ref_PyTorch_Compiled_Runtime_ms,Triton_Runtime_ms,Speedup,Speedup_vs_Compiled,Status,Error_Message,Timestamp
0,100_HingeLoss,level1,NVIDIA A100-SXM4-80GB,0.0,True,False,-1.0,-1.0,-1.0,-1.0,-1.0,SUCCESS,NaN,2025-11-09 22:16:03
1,10_3D_tensor_matrix_multiplication,level1,NVIDIA A100-SXM4-80GB,0.0,True,False,-1.0,-1.0,-1.0,-1.0,-1.0,SUCCESS,NaN,2025-11-09 22:16:06
2,11_4D_tensor_matrix_multiplication,level1,NVIDIA A100-SXM4-80GB,0.0,True,False,-1.0,-1.0,-1.0,-1.0,-1.0,SUCCESS,NaN,2025-11-09 22:16:11


In [4]:
df_L1 = df[df["Level"] == "level1"]
df_L2 = df[df["Level"] == "level2"]
df_L3 = df[df["Level"] == "level3"]

In [6]:
print(df_L1.shape)
print(df_L2.shape)
(print(df_L3.shape))

(100, 14)
(100, 14)
(50, 14)


In [10]:
print(df_L1["Compiled"].value_counts())
print(df_L1["Correctness"].value_counts())

Compiled
True     72
False    21
Name: count, dtype: int64
Correctness
False    86
True      7
Name: count, dtype: int64


In [11]:
print(df_L2["Compiled"].value_counts())
print(df_L2["Correctness"].value_counts())

Compiled
True     76
False    21
Name: count, dtype: int64
Correctness
False    96
True      1
Name: count, dtype: int64


In [12]:
print(df_L3["Compiled"].value_counts())
print(df_L3["Correctness"].value_counts())

Compiled
True     27
False     8
Name: count, dtype: int64
Correctness
False    35
Name: count, dtype: int64


In [ ]:
tmp = df_L1[df_L1["Correctness"] == True]
L1_speedup = tmp.Speedup.values
L1_compiled_speedup = tmp.Speedup_vs_Compiled.values

In [22]:
tmp = df_L2[df_L2["Correctness"] == True]
L2_speedup = tmp.Speedup.values
L2_compiled_speedup = tmp.Speedup_vs_Compiled.values

In [29]:
print(L1_speedup.mean())
print(L1_compiled_speedup.mean())

print(L2_speedup.mean())
print(L2_compiled_speedup.mean())

10.395714285714288
10.675714285714287
0.03
0.03


# Comprehensive Model Comparison
## Analysis across all models and levels


In [3]:
import os
import numpy as np
import pandas as pd

def analyze_model_csv(csv_path, model_name):
    """
    Analyze a model's CSV file and return statistics for each level.
    
    Args:
        csv_path: Path to the CSV file
        model_name: Name of the model for display
    
    Returns:
        DataFrame with statistics per level
    """
    if not os.path.exists(csv_path):
        print(f"File not found: {csv_path}")
        return None
    
    df = pd.read_csv(csv_path)
    
    results = []
    
    for level in ['level1', 'level2', 'level3']:
        df_level = df[df['Level'] == level]
        
        if len(df_level) == 0:
            continue
        
        # Count compiled and correctness
        total = len(df_level)
        compiled_count = df_level['Compiled'].sum()
        correct_count = df_level['Correctness'].sum()
        
        # Calculate speedup statistics for correct results only
        correct_df = df_level[df_level['Correctness'] == True]
        
        if len(correct_df) > 0:
            mean_speedup = correct_df['Speedup'].mean()
            median_speedup = correct_df['Speedup'].median()
            mean_speedup_compiled = correct_df['Speedup_vs_Compiled'].mean()
            median_speedup_compiled = correct_df['Speedup_vs_Compiled'].median()
        else:
            mean_speedup = 0.0
            median_speedup = 0.0
            mean_speedup_compiled = 0.0
            median_speedup_compiled = 0.0
        
        results.append({
            'Model': model_name,
            'Level': level.replace('level', 'L'),
            'Total': total,
            'Compiled': f"{compiled_count}/{total} ({100*compiled_count/total:.1f}%)",
            'Correct': f"{correct_count}/{total} ({100*correct_count/total:.1f}%)",
            'Mean_Speedup': f"{mean_speedup:.2f}x",
            'Median_Speedup': f"{median_speedup:.2f}x",
            'Mean_Speedup_vs_Compiled': f"{mean_speedup_compiled:.2f}x",
            'Median_Speedup_vs_Compiled': f"{median_speedup_compiled:.2f}x"
        })
    
    return pd.DataFrame(results)

def compare_all_models():
    """
    Compare all available model CSV files and return a comprehensive table.
    """
    models = [
        ('zeroshot_gpt_oss_20b.csv', 'GPT-OSS-20B'),
        ('zeroshot_qwen3_4b_base.csv', 'Qwen3-4B-Base'),
        ('zeroshot_qwen3_4b_finetuned.csv', 'Qwen3-4B-Finetuned'),
        ('zeroshot_qwen3_8b_base.csv', 'Qwen3-8B-Base'),
        ('zeroshot_qwen3_30b.csv', 'Qwen3-30B'),
        ('zeroshot_qwen3_235b_base.csv', 'Qwen3-235B'),
    ]
    
    all_results = []
    
    for csv_file, model_name in models:
        result = analyze_model_csv(csv_file, model_name)
        if result is not None:
            all_results.append(result)
    
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        return combined_df
    else:
        return None


In [4]:
# Run the comprehensive analysis
results_table = compare_all_models()

if results_table is not None:
    print("\n" + "="*120)
    print("COMPREHENSIVE MODEL COMPARISON ACROSS ALL LEVELS")
    print("="*120)
    print(results_table.to_string(index=False))
    print("="*120)
else:
    print("No results found")



COMPREHENSIVE MODEL COMPARISON ACROSS ALL LEVELS
             Model Level  Total       Compiled        Correct Mean_Speedup Median_Speedup Mean_Speedup_vs_Compiled Median_Speedup_vs_Compiled
       GPT-OSS-20B    L1    100 78/100 (78.0%) 11/100 (11.0%)        7.43x          1.07x                    7.06x                      1.01x
       GPT-OSS-20B    L2    100 87/100 (87.0%)   1/100 (1.0%)        1.00x          1.00x                    0.81x                      0.81x
       GPT-OSS-20B    L3     50  34/50 (68.0%)    3/50 (6.0%)        0.99x          0.98x                    0.74x                      0.78x
     Qwen3-4B-Base    L1    100 55/100 (55.0%) 12/100 (12.0%)        1.20x          1.00x                    0.93x                      0.98x
     Qwen3-4B-Base    L2    100 49/100 (49.0%)   3/100 (3.0%)        1.02x          1.01x                    0.73x                      0.71x
     Qwen3-4B-Base    L3     50  23/50 (46.0%)   8/50 (16.0%)        1.01x          1.00x         

In [ ]:
if results_table is not None:
    results_table.style.set_properties(**{
        'text-align': 'left',
        'font-size': '11px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('font-weight', 'bold'), ('background-color', '#f0f0f0')]}
    ])


## Summary Statistics by Model


In [6]:
# Calculate overall statistics per model
if results_table is not None:
    # Group by model and calculate aggregate statistics
    models = [
        ('zeroshot_gpt_oss_20b.csv', 'GPT-OSS-20B'),
        ('zeroshot_qwen3_4b_base.csv', 'Qwen3-4B-Base'),
        ('zeroshot_qwen3_4b_finetuned.csv', 'Qwen3-4B-Finetuned'),
        ('zeroshot_qwen3_8b_base.csv', 'Qwen3-8B-Base'),
        ('zeroshot_qwen3_30b.csv', 'Qwen3-30B'),
        ('zeroshot_qwen3_235b_base.csv', 'Qwen3-235B'),
    ]
    
    summary_data = []
    
    for csv_file, model_name in models:
        if not os.path.exists(csv_file):
            continue
        
        df = pd.read_csv(csv_file)
        
        total = len(df)
        compiled_count = df['Compiled'].sum()
        correct_count = df['Correctness'].sum()
        
        correct_df = df[df['Correctness'] == True]
        
        if len(correct_df) > 0:
            avg_speedup = correct_df['Speedup'].mean()
            avg_speedup_compiled = correct_df['Speedup_vs_Compiled'].mean()
        else:
            avg_speedup = 0.0
            avg_speedup_compiled = 0.0
        
        summary_data.append({
            'Model': model_name,
            'Total_Problems': total,
            'Compiled_Rate': f"{100*compiled_count/total:.1f}%",
            'Correctness_Rate': f"{100*correct_count/total:.1f}%",
            'Avg_Speedup': f"{avg_speedup:.2f}x",
            'Avg_Speedup_vs_Compiled': f"{avg_speedup_compiled:.2f}x"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n" + "="*100)
    print("OVERALL MODEL PERFORMANCE SUMMARY")
    print("="*100)
    print(summary_df.to_string(index=False))
    print("="*100)



OVERALL MODEL PERFORMANCE SUMMARY
             Model  Total_Problems Compiled_Rate Correctness_Rate Avg_Speedup Avg_Speedup_vs_Compiled
       GPT-OSS-20B             250         79.6%             6.0%       5.72x                   5.38x
     Qwen3-4B-Base             250         50.8%             9.2%       1.11x                   0.87x
Qwen3-4B-Finetuned             250          0.0%             0.0%       0.00x                   0.00x
     Qwen3-8B-Base             250         82.8%             7.6%       1.04x                   0.77x
         Qwen3-30B             250         70.0%             3.2%       9.10x                   9.34x
        Qwen3-235B             250         92.8%            12.8%       3.09x                   2.86x
